In [253]:
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

# NeuralNetwork

In [254]:
class NeuralNetwork:
    def __init__(self, n_hidden, n_output=1, lr=0.5, activation=np.arctan):
        self.lr = lr
        self.hidden_nodes = n_hidden
        self.output_nodes = n_output

        self.bias_h = np.random.uniform(-1, 1, (self.hidden_nodes, 1))
        self.bias_o = np.random.uniform(-1, 1, (self.output_nodes, 1))
        self.weights_ih = None
        self.weights_ho = None

    def sigmoid(self, x):
        return 1/(1 + np.exp(-x))

    def dsigmoid(self, x):
        return x * (1 - x)

    
    def feed_forward(self, inputs):
        inputs = np.array(inputs)
        self.hidden_sum = np.matmul(self.weights_ih, np.transpose(inputs)) + self.bias_h
        self.hidden_out = self.sigmoid(self.hidden_sum)
        self.output_sum = np.matmul(self.weights_ho, self.hidden_out) + self.bias_o
        self.output_out = self.sigmoid(self.output_sum)

        return np.transpose(self.output_out)

    
    def train(self, X, y):
        targets = np.array(y).reshape(1, -1)
        X = np.array(X)

        self.feed_forward(X)
        output_deltas = (y - self.output_out) * self.dsigmoid(self.output_out)
        hidden_errors = np.matmul(self.weights_ho.T, output_deltas)
        hidden_deltas = hidden_errors * self.dsigmoid(self.hidden_out)

        self.weights_ho += self.lr * np.matmul(output_deltas, self.hidden_out.T)
        self.weights_ih += self.lr * np.matmul(hidden_deltas, X)
        self.bias_h += self.lr * np.sum(hidden_deltas, axis=1, keepdims=True)
        self.bias_o += self.lr * np.sum(output_deltas, axis=1, keepdims=True)


    def fit(self, X, y):
        if self.weights_ih is None:
            self.input_nodes = len(X[0])
            self.weights_ih = np.random.uniform(-1, 1, (self.hidden_nodes, self.input_nodes))
            self.weights_ho = np.random.uniform(-1, 1, (self.output_nodes, self.hidden_nodes))
        
        for idx in range(len(y)):
            XX = np.array([X[idx]])
            yy = np.array([y[idx]])
            self.train(XX, yy)

In [255]:
nn = NeuralNetwork(n_hidden=2)

In [256]:
xor = np.array([
    [1, 0, 1],
    [0, 1, 1],
    [1, 1, 0],
    [0, 0, 0]
])
xor = np.tile(xor, (20000, 1))
np.random.shuffle(xor)
X = np.array(xor[:, :2])
y = np.array(xor[:, 2:3]).reshape(len(X))
nn.fit(X, y)

In [257]:
n = 4
X_test = np.array(xor[:n, :2])
y_test = np.array(xor[:n, 2:3]).reshape(n)


print(f"Inputs: {X_test}")
print(f"Labels: {y_test}")
nn.feed_forward(X_test)

Inputs: [[0 0]
 [0 1]
 [1 1]
 [0 0]]
Labels: [0 1 0 0]


array([[0.01151802],
       [0.98670145],
       [0.01031055],
       [0.01151802]])